In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

token = userdata.get("GITHUB_TOKEN")

repo_url = "https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env.git"
repo_dir = Path("/content/Phy_Informed_MIMO_RIS_Env")

askpass = tempfile.NamedTemporaryFile(
    mode="w",
    delete=False,
    suffix=".sh"
)
askpass.write("""#!/bin/sh
case "$1" in
    *Username*) echo "OsmanMertYilmaz" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""")
askpass.close()
os.chmod(askpass.name, 0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = askpass.name
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    if (repo_dir / ".git").exists():
        print("Repo var -> git pull")
        subprocess.run(
            ["git", "-C", str(repo_dir), "pull", "--ff-only"],
            env=env,
            check=True,
        )
    else:
        print("Repo yok -> git clone")
        subprocess.run(
            ["git", "clone", repo_url, str(repo_dir)],
            env=env,
            check=True,
        )
finally:
    os.remove(askpass.name)

print("GitHub sync tamamlandı.")


In [3]:
%cd /content/Phy_Informed_MIMO_RIS_Env
!pip install -e .
!pytest -q

/content/Phy_Informed_MIMO_RIS_Env
Obtaining file:///content/Phy_Informed_MIMO_RIS_Env
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for phy-informed-mimo-ris-env (pyproject.toml) ... done
  Created wheel for phy-informed-mimo-ris-env: filename=phy_informed_mimo_ris_env-0.1.0-0.editable-py3-none-any.whl size=1562 sha256=e1b32cedcf012bfe85f74a668ccf7acc8d20a6603d1be4fada913060a3dde466
  Stored in directory: /tmp/pip-ephem-wheel-cache-r3dsq693/wheels/8a/d3/e9/8beed57e2109a6ffc23a03c7789221e086e9f0aed17d22be5b
Successfully built phy-informed-mimo-ris-env
.....                                                                    [100%]
5 passed in 6.02s


In [4]:
from pathlib import Path

ENV_DIR = Path("/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments")
ENV_DIR.mkdir(parents=True, exist_ok=True)

print(ENV_DIR)

/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments


In [5]:
%cd /content/Phy_Informed_MIMO_RIS_Env

!python scripts/generate_environments.py \
    --config configs/nn_dataset_4000.yaml \
    --output /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv

/content/Phy_Informed_MIMO_RIS_Env
CONTROLLED NN ENVIRONMENT DATASET
Output              : /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv
Banks               : 4,000

Split counts
split
train                 2800
validation             600
test_interpolation     600

Scenario family counts
family
Indoor-Office    1000
RMa              1000
UMa              1000
UMi              1000

nRIS counts
nRIS
64     1000
128    1000
256    1000
512    1000

nT counts
nT
4     1000
8     2000
16    1000

Distance summary [m]
       family  banks  BR_min  BR_median   BR_max  RU_min  RU_median  RU_max  eta_min  eta_median  eta_max
Indoor-Office   1000   5.200     25.000   44.800   3.160     19.000  34.840    0.133       0.567    0.933
          RMa   1000 155.250    675.000 1194.750  53.750    425.000 796.250    0.167       0.613    0.956
          UMa   1000  72.150    285.000  497.850  36.325    167.500 298.675    0.199       0.628    0.931
          UMi   10

In [6]:
!python scripts/audit_environments.py \
    --config configs/nn_dataset_4000.yaml \
    /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv \
    --json-out /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000_audit.json

ENVIRONMENT DATASET AUDIT
CSV                         : /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv
Banks                       : 4,000
nT minimum                  : 4
Exact split geometry overlap: 0
Interpolation failures      : 0
Missing train geometry cells: 0

Split counts
split
train                 2800
validation             600
test_interpolation     600

Scenario x split
split          test_interpolation  train  validation
family                                              
Indoor-Office                 150    700         150
RMa                           150    700         150
UMa                           150    700         150
UMi                           150    700         150

LOS/NLOS pair x scenario
link_state     LOS_LOS  LOS_NLOS  NLOS_LOS  NLOS_NLOS
family                                               
Indoor-Office      250       250       250        250
RMa                250       250       250        250
UMa              

## 7. One-bank full teacher smoke test

Bu test gerçek production zincirini tek bir ağır bank üzerinde çalıştırır:

\[
32W \times 512Z \times 64000\ MC = 16384\ q05_{GG}\ label.
\]

`--select worst_case`, environment CSV içinden önce `nRIS=512`, sonra en büyük
`nT` ve `nR` bankını seçer.

In [ ]:
from pathlib import Path

SMOKE_DIR = Path("/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/smoke")
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

SMOKE_OUT = SMOKE_DIR / "teacher_smoke_bank.parquet"
print(SMOKE_OUT)

In [ ]:
%cd /content/Phy_Informed_MIMO_RIS_Env

!python scripts/smoke_test_teacher_bank.py \
    --config configs/nn_dataset_4000.yaml \
    --environments /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/environments/environments_4000.csv \
    --select worst_case \
    --n-mc 64000 \
    --output /content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/smoke/teacher_smoke_bank.parquet

In [ ]:
import pandas as pd

smoke = pd.read_parquet(
    "/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env/smoke/teacher_smoke_bank.parquet"
)

print("Rows:", len(smoke))
print("Banks:", smoke["bankID"].nunique())
print("W:", smoke[["WIdx_i11","WIdx_i12","WIdx_i2"]].drop_duplicates().shape[0])
print("Z:", smoke["zString"].nunique())
display(smoke.head())

assert len(smoke) == 32 * 512
assert smoke["bankID"].nunique() == 1
assert smoke[["WIdx_i11","WIdx_i12","WIdx_i2"]].drop_duplicates().shape[0] == 32
assert smoke["zString"].nunique() == 512

print("NOTEBOOK SMOKE OUTPUT PASS")